# ApexID baseline evaluation — MegaDescriptor + MiewIDv3

Open-set **validation** metrics on the cluster-branch train/val splits, comparable to ApexID training W&B logs.

**Kaggle setup**
- GPU: 16 GB VRAM (T4 x1 / P100)
- Dataset: `seifelkerdany/apexid-dataset`
- Secret: `GITHUB_TOKEN` (read access to `MIA-AI-Team/ApexID`, branch `cluster`)

Metrics use cloned ApexID `evaluate_all_validation_splits` (cosine gallery matching + NormAcc threshold on val).

In [1]:
%pip install -q git+https://github.com/WildlifeDatasets/wildlife-datasets@develop
%pip install -q git+https://github.com/WildlifeDatasets/wildlife-tools
%pip install -q git+https://github.com/WildMeOrg/wbia-plugin-miew-id.git
%pip install -q lmdb "transformers==4.45.2" timm albumentations scikit-learn pandas numpy torch networkx tqdm opencv-python-headless

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.1/141.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 77.3 MB/s eta 0:00:0

In [2]:
import os
import subprocess
import sys
from pathlib import Path

ON_KAGGLE = Path("/kaggle/input").exists()

if ON_KAGGLE:
    WORK_DIR = Path("/kaggle/working")
    REPO_PATH = WORK_DIR / "ApexID"
else:
    candidate = Path.cwd().resolve()
    if (candidate / "apexid").is_dir():
        REPO_PATH = candidate
    elif (candidate.parent / "apexid").is_dir():
        REPO_PATH = candidate.parent
    else:
        REPO_PATH = candidate / "ApexID"
    WORK_DIR = REPO_PATH.parent

if not (REPO_PATH / "apexid").is_dir():
    token = os.environ.get("GITHUB_TOKEN")
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient

            token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        except Exception as exc:
            raise RuntimeError("Set GITHUB_TOKEN as a Kaggle secret.") from exc
    REPO_PATH.parent.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(
        [
            "git",
            "clone",
            "--branch",
            "cluster",
            f"https://{token}@github.com/MIA-AI-Team/ApexID.git",
            str(REPO_PATH),
        ]
    )

if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))

print("Repo:", REPO_PATH.resolve())

Cloning into '/kaggle/working/ApexID'...


Repo: /kaggle/working/ApexID


In [3]:
if ON_KAGGLE:
    DATA_ROOT = Path("/kaggle/input/datasets/seifelkerdany/apexid-dataset/apexid-dataset")
else:
    DATA_ROOT = Path(os.environ.get("APEXID_DATA_ROOT", REPO_PATH.parent / "apexid-dataset"))

SPLIT_DIR = WORK_DIR / "open_set_splits"
METADATA_CSV = DATA_ROOT / "metadata.csv"
EMBED_CACHE_DIR = WORK_DIR / "embed_cache"
OUTPUT_DIR = WORK_DIR / "baseline_outputs"

for path in (SPLIT_DIR, EMBED_CACHE_DIR, OUTPUT_DIR):
    path.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT:", DATA_ROOT)
print("SPLIT_DIR:", SPLIT_DIR)
print("metadata exists:", METADATA_CSV.is_file())

DATA_ROOT: /kaggle/input/datasets/seifelkerdany/apexid-dataset/apexid-dataset
SPLIT_DIR: /kaggle/working/open_set_splits
metadata exists: True


In [4]:
required_splits = ["train.csv", "val_known.csv", "val_unknown.csv", "val.csv"]
missing = [name for name in required_splits if not (SPLIT_DIR / name).is_file()]

if missing:
    print("Regenerating splits (missing:", missing, ")")
    split_script = REPO_PATH / "data" / "make_open_set_splits.py"
    subprocess.check_call(
        [
            sys.executable,
            str(split_script),
            "--source",
            str(METADATA_CSV),
            "--output-dir",
            str(SPLIT_DIR),
            "--skip-image-check",
        ]
    )
else:
    print("Split CSVs already present in", SPLIT_DIR)

for name in required_splits:
    path = SPLIT_DIR / name
    rows = sum(1 for _ in open(path, encoding="utf-8")) - 1 if path.is_file() else 0
    print(f"{name}: {rows} rows")

Regenerating splits (missing: ['train.csv', 'val_known.csv', 'val_unknown.csv', 'val.csv'] )
train: rows=93101 identities=11677
train: top_datasets=[('HappyWhale', 28608), ('CzechLynx', 27839), ('humpback-whale-identification', 9617), ('WhaleSharkID', 3889), ('IPanda50', 3604), ('LeopardID2022', 3528), ('GiraffeZebraID', 2998), ('BalearicLizard', 2952)]
val_known: rows=25890 identities=5342
val_known: top_datasets=[('CzechLynx', 8109), ('HappyWhale', 7544), ('humpback-whale-identification', 2357), ('WhaleSharkID', 1139), ('IPanda50', 1047), ('LeopardID2022', 1026), ('BalearicLizard', 887), ('GiraffeZebraID', 833)]
val_unknown: rows=13542 identities=1671
val_unknown: top_datasets=[('HappyWhale', 5255), ('CzechLynx', 1881), ('humpback-whale-identification', 1650), ('WhaleSharkID', 930), ('BalearicLizard', 520), ('IPanda50', 516), ('GiraffeZebraID', 502), ('CTai', 456)]
val: rows=39432 identities=7013
val: top_datasets=[('HappyWhale', 12799), ('CzechLynx', 9990), ('humpback-whale-identifi

In [5]:
import torch

MEGA_NAME = "hf-hub:BVRA/MegaDescriptor-L-384"  # or hf-hub:BVRA/MegaDescriptor-T-224
MIEW_HF_TAG = "conservationxlabs/miewid-msv3"
BATCH_SIZE_MEGA = 64
BATCH_SIZE_MIEW = 64
USE_FP16 = True
NUM_WORKERS = 4 if ON_KAGGLE else 0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
UNKNOWN_LABEL = -100
CLEAR_EVERY_N_BATCHES = 50
DEBUG_VRAM = False

print("Device:", DEVICE)
print("MEGA_NAME:", MEGA_NAME)
print("MIEW_HF_TAG:", MIEW_HF_TAG)

Device: cuda
MEGA_NAME: hf-hub:BVRA/MegaDescriptor-L-384
MIEW_HF_TAG: conservationxlabs/miewid-msv3


In [6]:
import gc


def cuda_clear(label=""):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        if hasattr(torch.cuda, "ipc_collect"):
            torch.cuda.ipc_collect()
    if label:
        if torch.cuda.is_available():
            alloc = torch.cuda.memory_allocated() / 1e9
            peak = torch.cuda.max_memory_allocated() / 1e9
            print(f"[cuda_clear {label}] allocated={alloc:.2f} GB peak={peak:.2f} GB")
        else:
            print(f"[cuda_clear {label}] cpu mode")


def reset_peak_vram():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()


cuda_clear("init")

[cuda_clear init] allocated=0.00 GB peak=0.00 GB


In [7]:
import json
import math

import numpy as np
import pandas as pd
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from apexid.data.datasets.dataset import ApexID_Dataset
from apexid.evaluation import evaluate_all_validation_splits
from apexid.utils.seed import DEFAULT_SEED, make_torch_generator, seed_everything, seed_worker

seed_everything(DEFAULT_SEED)

EVAL_CFG = {
    "compute_known": True,
    "compute_unknown": True,
    "compute_rejection": True,
    "compute_mixed": True,
    "diagnostics_every_n_epochs": 1,
    "diagnostics": {"enabled": True, "max_pairs": 100000},
    "cmc_ranks": [20, 50],
    "sample_limits": {"gallery": None, "known": None, "unknown": None, "mixed": None},
    "top_k": [1, 5, 10],
    "known_unknown_threshold": None,
    "clustering": {"method": "threshold_connected_components", "threshold": 0.5},
    "rejection": {"fpr95_target_tpr": 0.95},
    "dir": {"far_levels": [0.01, 0.001]},
}

In [8]:
def read_frame(path):
    return ApexID_Dataset._standardize_columns(path).reset_index(drop=True)


def build_metrics_label_map(*frames):
    metric_frame = pd.concat(frames, ignore_index=True)
    return ApexID_Dataset.build_label_mapping(metric_frame)


train_frame = read_frame(SPLIT_DIR / "train.csv")
val_known_frame = read_frame(SPLIT_DIR / "val_known.csv")
val_unknown_frame = read_frame(SPLIT_DIR / "val_unknown.csv")
val_mixed_frame = read_frame(SPLIT_DIR / "val.csv")

print(
    "Loaded splits:",
    f"train={len(train_frame)}",
    f"val_known={len(val_known_frame)}",
    f"val_unknown={len(val_unknown_frame)}",
    f"val={len(val_mixed_frame)}",
)

Loaded splits: train=93101 val_known=25890 val_unknown=13542 val=39432


In [9]:
label_to_idx = ApexID_Dataset.build_label_mapping(train_frame)


def make_loader(frame, transform, batch_size, shuffle=False):
    metrics_label_map = build_metrics_label_map(
        train_frame,
        val_known_frame,
        val_unknown_frame,
        val_mixed_frame,
        frame,
    )
    dataset = ApexID_Dataset(
        frame,
        transform=transform,
        label_to_idx=label_to_idx,
        root_dir=str(DATA_ROOT),
        allow_unknown_identity=True,
        unknown_label=UNKNOWN_LABEL,
        metrics_label_map=metrics_label_map,
    )
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        worker_init_fn=seed_worker,
        generator=make_torch_generator(DEFAULT_SEED),
    )


def cache_paths(model_name, split_name):
    folder = EMBED_CACHE_DIR / model_name
    folder.mkdir(parents=True, exist_ok=True)
    return folder / f"{split_name}_emb.npy", folder / f"{split_name}_metric.npy", folder / f"{split_name}_cls.npy"


def save_embed_cache(model_name, split_name, embeddings, metric_labels, cls_labels):
    emb_path, metric_path, cls_path = cache_paths(model_name, split_name)
    np.save(emb_path, embeddings)
    np.save(metric_path, metric_labels)
    np.save(cls_path, cls_labels)


def load_embed_cache(model_name, split_name):
    emb_path, metric_path, cls_path = cache_paths(model_name, split_name)
    if not emb_path.is_file():
        return None
    return {
        "embeddings": np.load(emb_path),
        "metric_labels": np.load(metric_path),
        "cls_labels": np.load(cls_path),
    }

In [10]:
class EmbeddingModel(torch.nn.Module):
    """Wrap timm or HF models; return L2-normalized embeddings."""

    def __init__(self, core):
        super().__init__()
        self.core = core

    def forward(self, x):
        out = self.core(x)
        if isinstance(out, dict):
            if "embeddings" in out:
                out = out["embeddings"]
            elif "last_hidden_state" in out:
                out = out["last_hidden_state"].mean(dim=1)
            else:
                out = next(iter(out.values()))
        if out.ndim > 2:
            out = out.reshape(out.shape[0], -1)
        return F.normalize(out, dim=1)


@torch.inference_mode()
def extract_embeddings(model, loader, desc="Embeddings", use_fp16=USE_FP16):
    model.eval()
    all_embeddings = []
    all_metric_labels = []
    all_cls_labels = []

    autocast_ctx = torch.autocast("cuda", dtype=torch.float16) if use_fp16 and torch.cuda.is_available() else torch.autocast("cpu", enabled=False)

    with autocast_ctx:
        for batch_idx, batch in enumerate(tqdm(loader, desc=desc, leave=True)):
            images, cls_labels, metric_labels = batch
            images = images.to(DEVICE, non_blocking=True)
            embeddings = model(images).float().cpu().numpy()
            all_embeddings.append(embeddings)
            all_metric_labels.append(metric_labels.numpy())
            all_cls_labels.append(cls_labels.numpy())
            if batch_idx and batch_idx % CLEAR_EVERY_N_BATCHES == 0:
                cuda_clear(f"{desc} batch {batch_idx}")

    result = {
        "embeddings": np.concatenate(all_embeddings, axis=0) if all_embeddings else np.empty((0, 0)),
        "metric_labels": np.concatenate(all_metric_labels, axis=0) if all_metric_labels else np.empty(0, dtype=np.int64),
        "cls_labels": np.concatenate(all_cls_labels, axis=0) if all_cls_labels else np.empty(0, dtype=np.int64),
    }
    cuda_clear(desc)
    return result


def extract_split(model, model_name, split_name, frame, transform, batch_size, desc=None, use_cache=True):
    if use_cache:
        cached = load_embed_cache(model_name, split_name)
        if cached is not None:
            print(f"Loaded cache: {model_name}/{split_name}")
            return cached
    loader = make_loader(frame, transform, batch_size=batch_size)
    result = extract_embeddings(model, loader, desc=desc or f"{model_name}/{split_name}")
    save_embed_cache(model_name, split_name, result["embeddings"], result["metric_labels"], result["cls_labels"])
    return result

In [11]:
def scalar_metrics(raw_metrics):
    out = {}
    for key, value in raw_metrics.items():
        if torch.is_tensor(value):
            continue
        try:
            value = float(value)
        except (TypeError, ValueError):
            continue
        if math.isfinite(value):
            out[key] = value
    return out


def run_open_set_metrics(gallery, known, unknown, mixed, eval_cfg, split_label="val"):
    metrics = evaluate_all_validation_splits(
        gallery["embeddings"],
        gallery["metric_labels"],
        known["embeddings"],
        known["metric_labels"],
        unknown["embeddings"],
        unknown["metric_labels"],
        mixed["embeddings"],
        mixed["metric_labels"],
        known_cls_labels=known["cls_labels"],
        unknown_cls_labels=unknown["cls_labels"],
        mixed_cls_labels=mixed["cls_labels"],
        config=eval_cfg,
        epoch_index=0,
        classification_ignore_index=UNKNOWN_LABEL,
        chunk_size=1024,
        random_state=DEFAULT_SEED,
    )
    cleaned = scalar_metrics(metrics)
    print(f"[{split_label}] mixed/NormAcc={cleaned.get('mixed/NormAcc', float('nan')):.4f} threshold={cleaned.get('mixed/known_unknown_threshold', float('nan')):.4f}")
    return cleaned

## MegaDescriptor

In [12]:
import albumentations as A
import timm
from albumentations.pytorch import ToTensorV2

reset_peak_vram()
mega_core = timm.create_model(MEGA_NAME, num_classes=0, pretrained=True)
mega_model = EmbeddingModel(mega_core).to(DEVICE).eval()

data_cfg = timm.data.resolve_model_data_config(mega_core)
input_size = data_cfg.get("input_size", (3, 384, 384))
img_size = input_size[-1]
mean = data_cfg.get("mean", (0.485, 0.456, 0.406))
std = data_cfg.get("std", (0.229, 0.224, 0.225))

mega_transform = A.Compose([
    A.Resize(img_size, img_size),
    A.Normalize(mean=mean, std=std),
    ToTensorV2(),
])

cuda_clear("mega_loaded")
print("MegaDescriptor input size:", img_size)

config.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.94G [00:00<?, ?B/s]

[cuda_clear mega_loaded] allocated=0.80 GB peak=0.80 GB
MegaDescriptor input size: 384


In [13]:
mega_gallery = extract_split(
    mega_model, "megadescriptor", "gallery_train", train_frame, mega_transform, BATCH_SIZE_MEGA, desc="Mega/gallery"
)
mega_val_known = extract_split(
    mega_model, "megadescriptor", "val_known", val_known_frame, mega_transform, BATCH_SIZE_MEGA, desc="Mega/val_known"
)
mega_val_unknown = extract_split(
    mega_model, "megadescriptor", "val_unknown", val_unknown_frame, mega_transform, BATCH_SIZE_MEGA, desc="Mega/val_unknown"
)
mega_val_mixed = extract_split(
    mega_model, "megadescriptor", "val_mixed", val_mixed_frame, mega_transform, BATCH_SIZE_MEGA, desc="Mega/val_mixed"
)

mega_val_metrics = run_open_set_metrics(
    mega_gallery, mega_val_known, mega_val_unknown, mega_val_mixed, EVAL_CFG, split_label="Mega/val"
)
print("Mega val threshold:", mega_val_metrics["mixed/known_unknown_threshold"])

Mega/gallery:   0%|          | 0/1455 [00:00<?, ?it/s]

[cuda_clear Mega/gallery batch 50] allocated=1.33 GB peak=8.58 GB
[cuda_clear Mega/gallery batch 100] allocated=1.33 GB peak=8.58 GB
[cuda_clear Mega/gallery batch 150] allocated=1.33 GB peak=8.58 GB
[cuda_clear Mega/gallery batch 200] allocated=1.33 GB peak=8.58 GB
[cuda_clear Mega/gallery batch 250] allocated=1.33 GB peak=8.58 GB
[cuda_clear Mega/gallery batch 300] allocated=1.33 GB peak=8.58 GB
[cuda_clear Mega/gallery batch 350] allocated=1.33 GB peak=8.58 GB
[cuda_clear Mega/gallery batch 400] allocated=1.33 GB peak=8.58 GB
[cuda_clear Mega/gallery batch 450] allocated=1.33 GB peak=8.58 GB
[cuda_clear Mega/gallery batch 500] allocated=1.33 GB peak=8.58 GB
[cuda_clear Mega/gallery batch 550] allocated=1.33 GB peak=8.58 GB
[cuda_clear Mega/gallery batch 600] allocated=1.33 GB peak=8.58 GB
[cuda_clear Mega/gallery batch 650] allocated=1.33 GB peak=8.58 GB
[cuda_clear Mega/gallery batch 700] allocated=1.33 GB peak=8.58 GB
[cuda_clear Mega/gallery batch 750] allocated=1.33 GB peak=8.58

Mega/val_known:   0%|          | 0/405 [00:00<?, ?it/s]

[cuda_clear Mega/val_known batch 50] allocated=1.33 GB peak=8.58 GB
[cuda_clear Mega/val_known batch 100] allocated=1.33 GB peak=8.58 GB
[cuda_clear Mega/val_known batch 150] allocated=1.33 GB peak=8.58 GB
[cuda_clear Mega/val_known batch 200] allocated=1.33 GB peak=8.58 GB
[cuda_clear Mega/val_known batch 250] allocated=1.33 GB peak=8.58 GB
[cuda_clear Mega/val_known batch 300] allocated=1.33 GB peak=8.58 GB
[cuda_clear Mega/val_known batch 350] allocated=1.33 GB peak=8.58 GB
[cuda_clear Mega/val_known batch 400] allocated=1.33 GB peak=8.58 GB
[cuda_clear Mega/val_known] allocated=0.87 GB peak=8.58 GB


Mega/val_unknown:   0%|          | 0/212 [00:00<?, ?it/s]

[cuda_clear Mega/val_unknown batch 50] allocated=1.33 GB peak=8.58 GB
[cuda_clear Mega/val_unknown batch 100] allocated=1.33 GB peak=8.58 GB
[cuda_clear Mega/val_unknown batch 150] allocated=1.33 GB peak=8.58 GB
[cuda_clear Mega/val_unknown batch 200] allocated=1.33 GB peak=8.58 GB
[cuda_clear Mega/val_unknown] allocated=0.87 GB peak=8.58 GB


Mega/val_mixed:   0%|          | 0/617 [00:00<?, ?it/s]

[cuda_clear Mega/val_mixed batch 50] allocated=1.33 GB peak=8.58 GB
[cuda_clear Mega/val_mixed batch 100] allocated=1.33 GB peak=8.58 GB
[cuda_clear Mega/val_mixed batch 150] allocated=1.33 GB peak=8.58 GB
[cuda_clear Mega/val_mixed batch 200] allocated=1.33 GB peak=8.58 GB
[cuda_clear Mega/val_mixed batch 250] allocated=1.33 GB peak=8.58 GB
[cuda_clear Mega/val_mixed batch 300] allocated=1.33 GB peak=8.58 GB
[cuda_clear Mega/val_mixed batch 350] allocated=1.33 GB peak=8.58 GB
[cuda_clear Mega/val_mixed batch 400] allocated=1.33 GB peak=8.58 GB
[cuda_clear Mega/val_mixed batch 450] allocated=1.33 GB peak=8.58 GB
[cuda_clear Mega/val_mixed batch 500] allocated=1.33 GB peak=8.58 GB
[cuda_clear Mega/val_mixed batch 550] allocated=1.33 GB peak=8.58 GB
[cuda_clear Mega/val_mixed batch 600] allocated=1.33 GB peak=8.58 GB
[cuda_clear Mega/val_mixed] allocated=0.82 GB peak=8.58 GB
[Mega/val] mixed/NormAcc=0.5152 threshold=0.6154
Mega val threshold: 0.6153955459594727


In [14]:
del mega_model, mega_core
cuda_clear("after_mega_unload")

[cuda_clear after_mega_unload] allocated=0.01 GB peak=8.58 GB


## MiewIDv3 (Hugging Face)

In [15]:
from transformers import AutoModel

reset_peak_vram()
miew_core = AutoModel.from_pretrained(MIEW_HF_TAG, trust_remote_code=True)
miew_model = EmbeddingModel(miew_core).to(DEVICE).eval()

try:
    from miewid.models import base_transform_global

    miew_transform = base_transform_global
except Exception:
    miew_img_size = 440
    miew_transform = A.Compose([
        A.Resize(miew_img_size, miew_img_size),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
    print("Using fallback MiewID transform (440px ImageNet norm)")

cuda_clear("miew_loaded")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

config.json:   0%|          | 0.00/528 [00:00<?, ?B/s]

configuration_miewid.py:   0%|          | 0.00/777 [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/conservationxlabs/miewid-msv3:
- configuration_miewid.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_miewid.py: 0.00B [00:00, ?B/s]

... (2 outputs left out)

model.safetensors:   0%|          | 0.00/206M [00:00<?, ?B/s]

Building Model Backbone for efficientnetv2_rw_m model
config.model_name efficientnetv2_rw_m
model_name efficientnetv2_rw_m


model.safetensors:   0%|          | 0.00/214M [00:00<?, ?B/s]

final_in_features 2152
Using fallback MiewID transform (440px ImageNet norm)
[cuda_clear miew_loaded] allocated=0.22 GB peak=0.22 GB


In [16]:
miew_gallery = extract_split(
    miew_model, "miewidv3", "gallery_train", train_frame, miew_transform, BATCH_SIZE_MIEW, desc="Miew/gallery"
)
miew_val_known = extract_split(
    miew_model, "miewidv3", "val_known", val_known_frame, miew_transform, BATCH_SIZE_MIEW, desc="Miew/val_known"
)
miew_val_unknown = extract_split(
    miew_model, "miewidv3", "val_unknown", val_unknown_frame, miew_transform, BATCH_SIZE_MIEW, desc="Miew/val_unknown"
)
miew_val_mixed = extract_split(
    miew_model, "miewidv3", "val_mixed", val_mixed_frame, miew_transform, BATCH_SIZE_MIEW, desc="Miew/val_mixed"
)

miew_val_metrics = run_open_set_metrics(
    miew_gallery, miew_val_known, miew_val_unknown, miew_val_mixed, EVAL_CFG, split_label="Miew/val"
)
print("Miew val threshold:", miew_val_metrics["mixed/known_unknown_threshold"])

Miew/gallery:   0%|          | 0/1455 [00:00<?, ?it/s]

[cuda_clear Miew/gallery batch 50] allocated=0.47 GB peak=1.74 GB
[cuda_clear Miew/gallery batch 100] allocated=0.47 GB peak=1.74 GB
[cuda_clear Miew/gallery batch 150] allocated=0.47 GB peak=1.74 GB
[cuda_clear Miew/gallery batch 200] allocated=0.47 GB peak=1.74 GB
[cuda_clear Miew/gallery batch 250] allocated=0.47 GB peak=1.74 GB
[cuda_clear Miew/gallery batch 300] allocated=0.47 GB peak=1.74 GB
[cuda_clear Miew/gallery batch 350] allocated=0.47 GB peak=1.74 GB
[cuda_clear Miew/gallery batch 400] allocated=0.47 GB peak=1.74 GB
[cuda_clear Miew/gallery batch 450] allocated=0.47 GB peak=1.74 GB
[cuda_clear Miew/gallery batch 500] allocated=0.47 GB peak=1.74 GB
[cuda_clear Miew/gallery batch 550] allocated=0.47 GB peak=1.74 GB
[cuda_clear Miew/gallery batch 600] allocated=0.47 GB peak=1.74 GB
[cuda_clear Miew/gallery batch 650] allocated=0.47 GB peak=1.74 GB
[cuda_clear Miew/gallery batch 700] allocated=0.47 GB peak=1.74 GB
[cuda_clear Miew/gallery batch 750] allocated=0.47 GB peak=1.74

Miew/val_known:   0%|          | 0/405 [00:00<?, ?it/s]

[cuda_clear Miew/val_known batch 50] allocated=0.47 GB peak=1.74 GB
[cuda_clear Miew/val_known batch 100] allocated=0.47 GB peak=1.74 GB
[cuda_clear Miew/val_known batch 150] allocated=0.47 GB peak=1.74 GB
[cuda_clear Miew/val_known batch 200] allocated=0.47 GB peak=1.74 GB
[cuda_clear Miew/val_known batch 250] allocated=0.47 GB peak=1.74 GB
[cuda_clear Miew/val_known batch 300] allocated=0.47 GB peak=1.74 GB
[cuda_clear Miew/val_known batch 350] allocated=0.47 GB peak=1.74 GB
[cuda_clear Miew/val_known batch 400] allocated=0.47 GB peak=1.74 GB
[cuda_clear Miew/val_known] allocated=0.30 GB peak=1.74 GB


Miew/val_unknown:   0%|          | 0/212 [00:00<?, ?it/s]

[cuda_clear Miew/val_unknown batch 50] allocated=0.47 GB peak=1.74 GB
[cuda_clear Miew/val_unknown batch 100] allocated=0.47 GB peak=1.74 GB
[cuda_clear Miew/val_unknown batch 150] allocated=0.47 GB peak=1.74 GB
[cuda_clear Miew/val_unknown batch 200] allocated=0.47 GB peak=1.74 GB
[cuda_clear Miew/val_unknown] allocated=0.31 GB peak=1.74 GB


Miew/val_mixed:   0%|          | 0/617 [00:00<?, ?it/s]

[cuda_clear Miew/val_mixed batch 50] allocated=0.47 GB peak=1.74 GB
[cuda_clear Miew/val_mixed batch 100] allocated=0.47 GB peak=1.74 GB
[cuda_clear Miew/val_mixed batch 150] allocated=0.47 GB peak=1.74 GB
[cuda_clear Miew/val_mixed batch 200] allocated=0.47 GB peak=1.74 GB
[cuda_clear Miew/val_mixed batch 250] allocated=0.47 GB peak=1.74 GB
[cuda_clear Miew/val_mixed batch 300] allocated=0.47 GB peak=1.74 GB
[cuda_clear Miew/val_mixed batch 350] allocated=0.47 GB peak=1.74 GB
[cuda_clear Miew/val_mixed batch 400] allocated=0.47 GB peak=1.74 GB
[cuda_clear Miew/val_mixed batch 450] allocated=0.47 GB peak=1.74 GB
[cuda_clear Miew/val_mixed batch 500] allocated=0.47 GB peak=1.74 GB
[cuda_clear Miew/val_mixed batch 550] allocated=0.47 GB peak=1.74 GB
[cuda_clear Miew/val_mixed batch 600] allocated=0.47 GB peak=1.74 GB
[cuda_clear Miew/val_mixed] allocated=0.24 GB peak=1.74 GB
[Miew/val] mixed/NormAcc=0.5691 threshold=0.5856
Miew val threshold: 0.5856325030326843


In [17]:
del miew_model, miew_core
cuda_clear("after_miew_unload")

[cuda_clear after_miew_unload] allocated=0.01 GB peak=1.74 GB


## Compare all metrics

In [18]:
ALL_RESULTS = {
    "MegaDescriptor": mega_val_metrics,
    "MiewIDv3": miew_val_metrics,
}

rows = []
for model_name, metrics in ALL_RESULTS.items():
    for metric_name, value in sorted(metrics.items()):
        rows.append({"model": model_name, "metric": metric_name, "value": value})

metrics_df = pd.DataFrame(rows)
print(f"Total metric rows: {len(metrics_df)}")
metrics_df.head(10)

Total metric rows: 88


,model,metric,value
0,MegaDescriptor,diagnostics/alignment,1.248372e+00
1,MegaDescriptor,diagnostics/embedding_norm_max,1.000000e+00
2,MegaDescriptor,diagnostics/embedding_norm_mean,1.000000e+00
3,MegaDescriptor,diagnostics/embedding_norm_min,1.000000e+00
4,MegaDescriptor,diagnostics/embedding_norm_std,1.678825e-08
5,MegaDescriptor,diagnostics/inter_similarity,3.909699e-02
6,MegaDescriptor,diagnostics/intra_inter_gap,3.367170e-01
7,MegaDescriptor,diagnostics/intra_similarity,3.758140e-01
8,MegaDescriptor,diagnostics/knn_top1,4.957126e-01
9,MegaDescriptor,diagnostics/knn_top5,6.185400e-01


In [19]:
KEY_METRICS = [
    "known/mAP", "known/top1", "known/top5", "known/top10", "known/BAKS",
    "unknown/ARI", "unknown/AMI", "unknown/v_measure", "unknown/BAUS",
    "rejection/AUROC", "rejection/FAR", "rejection/FPR95",
    "mixed/ARI", "mixed/F1", "mixed/NormAcc", "mixed/BAKS", "mixed/BAUS",
    "mixed/OSCR_AUC", "mixed/DIR_at_1FAR", "mixed/DIR_at_0_1FAR",
    "mixed/known_unknown_threshold",
    "diagnostics/intra_inter_gap", "diagnostics/knn_top1", "diagnostics/knn_top5",
]

pivot = metrics_df.pivot_table(index="metric", columns="model", values="value", aggfunc="first")
available_keys = [key for key in KEY_METRICS if key in pivot.index]
pivot.loc[available_keys]

model,MegaDescriptor,MiewIDv3
metric,,
known/mAP,0.258258,0.244744
known/top1,0.495713,0.603940
known/top5,0.618540,0.717999
known/top10,0.673001,0.763538
known/BAKS,0.444210,0.563671
unknown/ARI,0.019681,0.021470
unknown/AMI,0.399412,0.407780
unknown/v_measure,0.732943,0.661626
unknown/BAUS,0.799733,0.735122


In [20]:
csv_path = OUTPUT_DIR / "baseline_mega_miew_metrics.csv"
json_path = OUTPUT_DIR / "baseline_metrics.json"

metrics_df.to_csv(csv_path, index=False)

report = {
    "models": ["MegaDescriptor", "MiewIDv3"],
    "mega_name": MEGA_NAME,
    "miew_hf_tag": MIEW_HF_TAG,
    "split_dir": str(SPLIT_DIR),
    "split": "val",
    "results": ALL_RESULTS,
}
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, sort_keys=True)

print("Saved:", csv_path)
print("Saved:", json_path)

Saved: /kaggle/working/baseline_outputs/baseline_mega_miew_metrics.csv
Saved: /kaggle/working/baseline_outputs/baseline_metrics.json
